# Text-to-SQL with Lineage-Aware Context

**Example: Schema-aware SQL generation from natural language**

`GenerateSQLTool` builds its prompt from the pipeline's lineage graph, not just a list of tables:

- **Schema with table roles** — `(Source table)` / `(Intermediate table)` / `(Final table)`, plus an instruction to prefer final tables
- **Table relationships and column lineage** — how outputs derive from sources
- **Join hints** — equi-joins *observed* in your pipeline's SQL, plus `candidate:` joins *inferred* between tables that share identity-preserving lineage (renames qualify; transforms, aggregates, and filters fail closed)

Requirements:
- Install: `uv pip install -e .`
- For Ollama: Install Ollama and run: `ollama pull gemma4:31b`

### Build a pipeline with real joins

Note the two marts are **never joined directly** in the pipeline, and `mart.customer_orders` renames the key to `cust_id` — the join hint between them must be *inferred* from shared lineage.

In [1]:
from clgraph import Pipeline
from clgraph.tools import ContextBuilder, ContextConfig
from clgraph.tools.sql import GenerateSQLTool

pipeline = Pipeline.from_dict(
    {
        "staging_orders": """
            CREATE TABLE staging.orders AS
            SELECT order_id, customer_id, amount, order_date
            FROM raw.orders
        """,
        "mart_customer_revenue": """
            CREATE TABLE mart.customer_revenue AS
            SELECT o.customer_id, c.region, SUM(o.amount) AS total_revenue
            FROM staging.orders o
            JOIN raw.customers c ON o.customer_id = c.id
            GROUP BY o.customer_id, c.region
        """,
        "mart_customer_orders": """
            CREATE TABLE mart.customer_orders AS
            SELECT customer_id AS cust_id, COUNT(*) AS order_count
            FROM staging.orders
            GROUP BY customer_id
        """,
    },
    dialect="bigquery",
)

builder = ContextBuilder(pipeline, ContextConfig())
for table in builder.resolve_context_tables():
    print(f"{table:30} {builder.table_role(table)}")

mart.customer_orders           final
mart.customer_revenue          final
staging.orders                 intermediate
raw.customers                  source
raw.orders                     source


### What the LLM receives

No LLM needed for this part — these are the graph-derived sections the tool assembles into every prompt. Note the two kinds of join hints: `observed in ...` (extracted from the pipeline's actual SQL) and `candidate:` (inferred — both columns pass through unchanged from the same source column, so the rename `cust_id` is still recognized).

In [2]:
tables = builder.resolve_context_tables()
print(builder.build_relationship_context(tables))
print()
print(builder.build_lineage_context(tables))
print()
print(builder.build_join_context(tables))

## Table Relationships

- staging.orders is derived from raw.orders
- mart.customer_revenue is derived from raw.customers
- mart.customer_revenue is derived from staging.orders
- mart.customer_orders is derived from staging.orders

## Column Lineage

- mart.customer_orders.cust_id <- raw.orders.customer_id
- mart.customer_orders.order_count <- raw.orders.order_id, raw.orders.customer_id, raw.orders.amount
- mart.customer_revenue.customer_id <- raw.orders.customer_id
- mart.customer_revenue.region <- raw.customers.region, raw.customers.id, raw.orders.customer_id
- mart.customer_revenue.total_revenue <- raw.orders.amount
- staging.orders.order_id <- raw.orders.order_id
- staging.orders.customer_id <- raw.orders.customer_id
- staging.orders.amount <- raw.orders.amount
- staging.orders.order_date <- raw.orders.order_date

## Join Hints

- staging.orders.customer_id = raw.customers.id (observed in mart_customer_revenue)
- candidate: mart.customer_orders.cust_id = mart.customer_revenue.custo

### Generate SQL (direct strategy)

The first question is answerable from a single final table — the role labels and prefer-final-tables instruction should steer the model to `mart.customer_revenue` rather than re-deriving from raw tables.

In [3]:
llm = None
try:
    from langchain_ollama import ChatOllama

    llm = ChatOllama(model="gemma4:31b", temperature=0.1)
    llm.invoke("Say OK.")  # connection check
    print("✅ Ollama connected (model: gemma4:31b)")
except Exception as e:
    llm = None
    print(f"⚠️  Ollama not available ({e}) - LLM cells will be skipped")

if llm:
    tool = GenerateSQLTool(pipeline, llm)
    result = tool.run(
        question="Which region generated the most revenue? Show region and total revenue.",
    )
    print()
    print(result.data["sql"])
    print()
    print("Explanation:", result.data["explanation"])

✅ Ollama connected (model: gemma4:31b)



SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM
    `mart.customer_revenue`
GROUP BY
    region
ORDER BY
    total_revenue DESC
LIMIT 1

Explanation: I will use the `mart.customer_revenue` final table as it already contains the pre-calculated revenue and the associated region for each customer. I will group the data by region, sum the total revenue, and order the results in descending order to identify the region with the highest revenue.


### A question that needs the *candidate* join

Revenue lives in one mart, order counts in the other, and the join key is renamed on one side. The `candidate:` hint tells the model exactly how to bridge them.

In [4]:
if llm:
    result = tool.run(
        question="For each customer, show their total revenue and their order count.",
        include_explanation=False,
    )
    print(result.data["sql"])
    print()
    print("Tables used:", result.data["tables_used"])

SELECT
    t1.customer_id,
    t1.total_revenue,
    t2.order_count
FROM
    `mart.customer_revenue` AS t1
JOIN
    `mart.customer_orders` AS t2
ON
    t1.customer_id = t2.cust_id

Tables used: ['mart.customer_orders', 'mart.customer_revenue', 'staging.orders', 'raw.customers', 'raw.orders']


### Two-stage strategy

`strategy="two_stage"` first asks the model which tables are relevant, expands the selection with lineage ancestors (transitively, default depth 2), then generates against that smaller context — useful for pipelines with many tables.

In [5]:
if llm:
    result = tool.run(
        question="For each customer, show their total revenue and their order count.",
        strategy="two_stage",
        include_explanation=False,
    )
    print(result.data["sql"])
    print()
    print("Tables used:", result.data["tables_used"])

SELECT
    t1.customer_id,
    t1.total_revenue,
    t2.order_count
FROM
    `mart.customer_revenue` AS t1
JOIN
    `mart.customer_orders` AS t2
ON
    t1.customer_id = t2.cust_id

Tables used: ['mart.customer_revenue', 'mart.customer_orders', 'raw.customers', 'staging.orders', 'raw.orders']


### Via the LineageAgent

The agent routes natural-language questions to the right tool automatically — SQL-generation questions land on `generate_sql`.

In [6]:
if llm:
    from clgraph.agent import LineageAgent

    agent = LineageAgent(pipeline, llm=llm)
    result = agent.query("Write SQL to show total revenue per region")
    print("Tool used:", result.tool_used)
    print()
    print(result.answer)

Tool used: generate_sql

To find the total revenue per region, I will use the `mart.customer_revenue` table, which contains both the regional information and the revenue associated with each customer. I will sum the `total_revenue` column and group the results by the `region` column.

```sql
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM
    `mart.customer_revenue`
GROUP BY
    region
```


### Tips

- `ContextConfig` controls the context budget: `max_tables`, `max_join_hints`, `max_lineage_lines`, `lineage_expansion_depth`, `annotate_table_roles`.
- Column descriptions (see `llm_description_generation.ipynb`) are included in the schema context when present — richer descriptions mean better SQL.
- Candidate joins are conservative by design: any transform, aggregate, or filter on a column's lineage path disqualifies it, so hints never equate columns whose values could differ.